In [2]:
import glob
import math
import os
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt
from sklearn.model_selection import GroupKFold

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cpu')

TARGET_FS_WORK = 10.0
LO_HZ, HI_HZ = 0.15, 4.6
N_LEVELS = 12
THRESHOLDS = (0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.15, 1.3)
BATCH = 48
OFFSET_GRID = np.arange(-4.0, 4.01, 0.25)
CFGS = [(ps, tol, gain) for ps in (0.0, 6.0) for tol in (0.01,) for gain in (32.0, 64.0)]


class LIFDetectors(nn.Module):
    def __init__(self, n_in, n_out, thresholds=THRESHOLDS, alpha_init=0.25, beta_init=0.35):
        super().__init__()
        w = torch.zeros(n_out, n_in)
        thr = torch.empty(n_out)
        for d in range(n_out):
            w[d, d % n_in] = 1.0
            thr[d] = thresholds[(d // n_in) % len(thresholds)]
        self.weight = nn.Parameter(w + 0.02 * torch.randn(n_out, n_in))
        self.bias = nn.Parameter(torch.zeros(n_out))
        self.thr_raw = nn.Parameter(torch.log(torch.expm1((thr - 0.05).clamp_min(1e-3))))
        self.a_raw = nn.Parameter(torch.full((n_out,), math.log(alpha_init / (1 - alpha_init))))
        self.b_raw = nn.Parameter(torch.full((n_out,), math.log(beta_init / (1 - beta_init))))

    @property
    def threshold(self):
        return F.softplus(self.thr_raw) + 0.05

    def membrane(self, x):
        a, b = torch.sigmoid(self.a_raw), torch.sigmoid(self.b_raw)
        drive = F.linear(x, self.weight, self.bias)
        syn = drive.new_zeros(drive.shape[0], drive.shape[2])
        mem = torch.zeros_like(syn)
        out = []
        for t in range(drive.shape[1]):
            syn = a * syn + (1 - a) * drive[:, t]
            mem = b * mem + (1 - b) * syn
            out.append(mem)
        return torch.stack(out, 1)

    @torch.no_grad()
    def forward(self, x, mask):
        a, b = torch.sigmoid(self.a_raw), torch.sigmoid(self.b_raw)
        thr = self.threshold
        drive = F.linear(x, self.weight, self.bias)
        B, T, D = drive.shape
        syn = drive.new_zeros(B, D)
        mem = drive.new_zeros(B, D)
        armed = drive.new_ones(B, D)
        total = drive.new_zeros(B, D)
        for t in range(T):
            syn = a * syn + (1 - a) * drive[:, t]
            mem = b * mem + (1 - b) * syn
            up = (mem > thr).float()
            s = armed * up
            re = ((-mem) > thr).float()
            armed = armed * (1 - up) + (1 - armed) * re
            total = total + s * mask[:, t].unsqueeze(-1)
        return total


def batch_tensors(flat, offs, lengths, sel, n_chan):
    span = int(lengths[sel].max())
    x = np.zeros((len(sel), span, n_chan), np.float32)
    for j, i in enumerate(sel):
        x[j, :lengths[i]] = flat[offs[i]:offs[i + 1]]
    n = torch.from_numpy(lengths[sel])
    return (torch.from_numpy(x).to(DEVICE),
            (torch.arange(span)[None, :] < n[:, None]).float().to(DEVICE))


def preprocess(flat, offsets, dec, n_in, fs, lo_hz=LO_HZ, hi_hz=HI_HZ):
    sos = butter(4, [lo_hz / (fs / 2), hi_hz / (fs / 2)], btype='band', output='sos')
    n_chan = n_in + 1
    parts, lengths = [], []
    for i in range(len(offsets) - 1):
        imu = flat[offsets[i]:offsets[i + 1]].astype(np.float64)
        x = sosfiltfilt(sos, imu, axis=0)
        x = x - x.mean(0)
        s = x.std(0)
        s[s < 1e-9] = 1.0
        x = x / s
        _, _, vt = np.linalg.svd(x - x.mean(0), full_matrices=False)
        comp = x @ vt[0]
        comp = comp / max(comp.std(), 1e-9)
        sig = np.concatenate([x, comp[:, None]], axis=1).astype(np.float32)
        k = (len(sig) // dec) * dec
        parts.append(sig[:k].reshape(-1, dec, n_chan).mean(1))
        lengths.append(len(parts[-1]))
    lengths = np.asarray(lengths, np.int64)
    offs = np.zeros(len(lengths) + 1, np.int64)
    offs[1:] = np.cumsum(lengths)
    return np.concatenate(parts, 0), offs, lengths


def counts_with_calibration(f, o, l, n_chan, n_det, calib_idx, target_idx):
    torch.manual_seed(SEED)
    model = LIFDetectors(n_chan, n_det).to(DEVICE)
    ci = calib_idx[np.argsort(l[calib_idx])]
    warm = ci[len(ci) // 2: len(ci) // 2 + BATCH]
    xw, mw = batch_tensors(f, o, l, warm, n_chan)
    model.eval()
    mem = model.membrane(xw)
    sd = mem[mw.bool()].std(0).clamp_min(1e-6)
    model.weight.data.div_(sd[:, None])
    model.bias.data.div_(sd)
    C = np.zeros((len(target_idx), n_det), np.float32)
    order = np.argsort(l[target_idx])
    for b in range(0, len(target_idx), BATCH):
        loc = order[b:b + BATCH]
        x, m = batch_tensors(f, o, l, target_idx[loc], n_chan)
        C[loc] = model(x, m).cpu().numpy()
    return C.astype(np.float64)


def vote(counts, tol, gain, log_prior=None):
    alive = counts > 0.5
    width = np.maximum(tol * counts, 0.5)
    diff = counts[:, :, None] - counts[:, None, :]
    agree = np.exp(-0.5 * (diff / width[:, :, None]) ** 2) * alive[:, None, :]
    s = gain * np.log(agree.sum(-1) + 1e-6)
    if log_prior is not None:
        s = s + log_prior(counts)
    s = np.where(alive, s, -1e9)
    s = s - s.max(1, keepdims=True)
    w = np.exp(s)
    w = w / np.maximum(w.sum(1, keepdims=True), 1e-12)
    return (w * counts).sum(1)


def make_log_prior(train_counts, scale):
    t = np.asarray(train_counts, np.float64)
    bw = max(1.06 * t.std() * len(t) ** (-0.2), 0.8)

    def lp(c):
        d = (c[:, :, None] - t[None, None, :]) / bw
        dens = np.exp(-0.5 * d ** 2).sum(-1) / (len(t) * bw)
        v = np.log(dens + 1e-12)
        return scale * (v - v.max(axis=1, keepdims=True))
    return lp


def score(p, t):
    e = np.rint(p).astype(int) - np.asarray(t)
    a = np.abs(e)
    return dict(exact=float(np.mean(e == 0)), w1=float(np.mean(a <= 1)),
                w2=float(np.mean(a <= 2)), mae=float(np.mean(a)),
                rmse=float(np.sqrt((e.astype(np.float64) ** 2).mean())), bias=float(np.mean(e)))


def infer_sessions(activity, raw_ids):
    order = np.argsort(raw_ids)
    groups = np.empty(len(activity), np.int64)
    cur, seen = 0, set()
    started = False
    for idx in order:
        ex = activity[idx]
        if started and ex in seen:
            cur += 1
            seen = set()
        groups[idx] = cur
        seen.add(ex)
        started = True
    return groups


def select_config(f, o, l, y, groups, n_chan, n_det, fit_idx, sel_idx):
    Cf = counts_with_calibration(f, o, l, n_chan, n_det, fit_idx, fit_idx)
    Cs = counts_with_calibration(f, o, l, n_chan, n_det, fit_idx, sel_idx)
    best, best_acc = None, -1.0
    for ps, tol, gain in CFGS:
        lp = make_log_prior(y[fit_idx], ps) if ps > 0 else None
        pf = vote(Cf, tol, gain, lp)
        off = float(max(OFFSET_GRID,
                        key=lambda o_: np.mean(np.rint(pf + o_).astype(int) == y[fit_idx])))
        a = float(np.mean(np.rint(vote(Cs, tol, gain, lp) + off).astype(int) == y[sel_idx]))
        if a > best_acc:
            best_acc, best = a, (ps, tol, gain)
    return best


def fit_predict(f, o, l, y, n_chan, n_det, tr, te, cfg):
    ps, tol, gain = cfg
    C_tr = counts_with_calibration(f, o, l, n_chan, n_det, tr, tr)
    C_te = counts_with_calibration(f, o, l, n_chan, n_det, tr, te)
    lp = make_log_prior(y[tr], ps) if ps > 0 else None
    p_tr = vote(C_tr, tol, gain, lp)
    off = float(max(OFFSET_GRID,
                    key=lambda o_: np.mean(np.rint(p_tr + o_).astype(int) == y[tr])))
    return vote(C_te, tol, gain, lp) + off


def inner_split(groups, tr, seed=SEED):
    tr_g = np.unique(groups[tr])
    rng = np.random.default_rng(seed)
    rng.shuffle(tr_g)
    hold = set(tr_g[:max(1, len(tr_g) // 4)])
    return (tr[~np.isin(groups[tr], list(hold))], tr[np.isin(groups[tr], list(hold))])


def run_leave_group_out(f, o, l, y, groups, n_chan, n_det, n_splits, seed=SEED):
    pred = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=n_splits).split(np.zeros(len(y)), groups=groups):
        ifit, isel = inner_split(groups, tr, seed)
        cfg = select_config(f, o, l, y, groups, n_chan, n_det, ifit, isel)
        pred[te] = fit_predict(f, o, l, y, n_chan, n_det, tr, te, cfg)
    return pred


def report(name, protocol, m):
    print(f'{name} | {protocol} | exact {m["exact"]:.2%} | +-1 {m["w1"]:.2%} | '
          f'+-2 {m["w2"]:.2%} | MAE {m["mae"]:.3f} | RMSE {m["rmse"]:.3f}')


In [3]:
# ##################### Microsoft 
FS_MS = 50.0
MAX_SECONDS = 120
FIXED_LENGTH_SAMPLES = int(FS_MS * MAX_SECONDS)
EXCLUDE_ACTIVITIES_MS = ('Fast Alternating Punches',)
NUM_FOLDS_MS = 7
DEC_MS = round(FS_MS / TARGET_FS_WORK)
PREFERRED = ('exercise_data.50.0000_singleonly.mat', 'single.mat')

mat_cands = sorted(Path('/kaggle/input').rglob('*.mat'))
DATASET_FILE_MS = next(p for nm in PREFERRED for p in mat_cands if p.name == nm)


def scalar(value, default=None):
    try:
        a = np.asarray(value).reshape(-1)
        if a.size != 1:
            return default
        x = float(a[0])
        return x if np.isfinite(x) else default
    except (TypeError, ValueError):
        return default


def clean_stream(matrix):
    a = np.asarray(matrix, dtype=np.float64)
    if a.ndim != 2 or a.shape[1] < 4:
        return None
    a = a[:, :4]
    a = a[np.all(np.isfinite(a), axis=1)]
    if len(a) < 2:
        return None
    a = a[np.argsort(a[:, 0], kind='stable')]
    _, u = np.unique(a[:, 0], return_index=True)
    a = a[np.sort(u)]
    return a if len(a) >= 2 and a[-1, 0] > a[0, 0] else None


def aligned_imu(accel, gyro):
    start, end = max(accel[0, 0], gyro[0, 0]), min(accel[-1, 0], gyro[-1, 0])
    if end <= start:
        return None
    n = int(np.floor((end - start) * FS_MS + 1e-6)) + 1
    if n <= 1 or n > FIXED_LENGTH_SAMPLES:
        return None
    grid = start + np.arange(n, dtype=np.float64) / FS_MS
    out = np.empty((n, 6), dtype=np.float32)
    for ax in range(3):
        out[:, ax] = np.interp(grid, accel[:, 0], accel[:, ax + 1])
        out[:, ax + 3] = np.interp(grid, gyro[:, 0], gyro[:, ax + 1])
    return out


mat = loadmat(DATASET_FILE_MS, squeeze_me=True, struct_as_record=False)
subject_data = np.asarray(mat['subject_data'], dtype=object)
activities = [str(x) for x in np.atleast_1d(mat['exerciseConstants'].activities)]

imus, counts_ms_all, activity_ms_all = [], [], []
for row in range(subject_data.shape[0]):
    for col in range(subject_data.shape[1]):
        cell = subject_data[row, col]
        if cell is None or (isinstance(cell, np.ndarray) and cell.size == 0):
            continue
        for rec in np.atleast_1d(cell).reshape(-1):
            if rec is None or not hasattr(rec, 'data'):
                continue
            reps = scalar(getattr(rec, 'activityReps', None))
            if reps is None or reps <= 0 or abs(reps - round(reps)) > 1e-6:
                continue
            if scalar(getattr(rec, 'incompleteData', 0), 0) != 0:
                continue
            d = rec.data
            accel = clean_stream(getattr(d, 'accelDataMatrix', None)) \
                if getattr(d, 'accelDataMatrix', None) is not None else None
            gyro = clean_stream(getattr(d, 'gyroDataMatrix', None)) \
                if getattr(d, 'gyroDataMatrix', None) is not None else None
            if accel is None or gyro is None:
                continue
            if min(accel[-1, 0], gyro[-1, 0]) - max(accel[0, 0], gyro[0, 0]) > \
                    MAX_SECONDS + 0.5 / FS_MS:
                continue
            imu = aligned_imu(accel, gyro)
            if imu is None:
                continue
            imus.append(imu)
            counts_ms_all.append(int(round(reps)))
            activity_ms_all.append(str(getattr(rec, 'activityName', activities[col])))

del mat, subject_data
counts_ms_all = np.asarray(counts_ms_all, dtype=np.int64)
activity_ms_all = np.asarray(activity_ms_all)

keep_ms = np.array([a not in EXCLUDE_ACTIVITIES_MS for a in activity_ms_all])
imus = [im for im, k in zip(imus, keep_ms) if k]
y_ms = counts_ms_all[keep_ms]
act_ms = activity_ms_all[keep_ms]
lengths_ms = np.asarray([len(im) for im in imus], np.int64)
offsets_ms = np.zeros(len(imus) + 1, np.int64)
offsets_ms[1:] = np.cumsum(lengths_ms)
flat_ms = np.concatenate(imus, 0)
n_in_ms = flat_ms.shape[1]
n_chan_ms = n_in_ms + 1
n_det_ms = n_chan_ms * N_LEVELS

f_ms, o_ms, l_ms = preprocess(flat_ms, offsets_ms, DEC_MS, n_in_ms, FS_MS)
pred_ms = run_leave_group_out(f_ms, o_ms, l_ms, y_ms, act_ms, n_chan_ms, n_det_ms,
                              n_splits=NUM_FOLDS_MS)
m_ms = score(pred_ms, y_ms)

print(f'{len(y_ms)} recordings | {len(set(act_ms))} activities | {n_det_ms} detectors')
report('Microsoft/RecoFit', 'leave-activity-out', m_ms)


1294 recordings | 48 activities | 84 detectors
Microsoft/RecoFit | leave-activity-out | exact 59.66% | +-1 82.38% | +-2 85.86% | MAE 1.985 | RMSE 5.212


In [4]:
# ############# Crossfit -- leave-session-out (Soro et al. comparison protocol)
FS_HAR = 100.0
ACC_GYR = [0, 1, 2, 3, 4, 5, 9, 10, 11, 12, 13, 14]
EXCLUDE_EXERCISES_HAR = ('Null',)
MIN_SAMPLES_HAR = 200
DEC_HAR = round(FS_HAR / TARGET_FS_WORK)

DATA_ROOT_HAR = str(next(p for p in sorted(Path('/kaggle/input').rglob('preprocessed_numpy_data'))
                         if p.is_dir()))


def load_har(root, channels=ACC_GYR):
    ex_root = os.path.join(root, 'np_exercise_data')
    rep_root = os.path.join(root, 'np_reps_data')
    exercises = sorted(d for d in os.listdir(ex_root)
                       if os.path.isdir(os.path.join(ex_root, d)))
    parts, lengths, counts, acts, sids = [], [], [], [], []
    for ex in exercises:
        if ex in EXCLUDE_EXERCISES_HAR:
            continue
        for setf in sorted(glob.glob(os.path.join(ex_root, ex, '*.npy'))):
            sid = os.path.basename(setf)[:-4]
            reps = glob.glob(os.path.join(rep_root, ex, f'{sid}_*.npy'))
            if len(reps) < 1:
                continue
            a = np.load(setf)
            if a.ndim != 2 or a.shape[0] < 18 or a.shape[1] < MIN_SAMPLES_HAR:
                continue
            sig = a[channels].T.astype(np.float32)
            if not np.all(np.isfinite(sig)):
                sig = np.nan_to_num(sig, nan=0.0, posinf=0.0, neginf=0.0)
            parts.append(sig)
            lengths.append(len(sig))
            counts.append(len(reps))
            acts.append(ex)
            sids.append(int(sid.split('_')[-1]))
    lengths = np.asarray(lengths, np.int64)
    offsets = np.zeros(len(lengths) + 1, np.int64)
    offsets[1:] = np.cumsum(lengths)
    return dict(flat=np.concatenate(parts, 0), offsets=offsets, lengths=lengths,
                counts=np.asarray(counts, np.int64), activity=np.asarray(acts),
                session=np.asarray(sids, np.int64))


RAW_HAR = load_har(DATA_ROOT_HAR)
flat_har, offsets_har = RAW_HAR['flat'], RAW_HAR['offsets']
y_har = RAW_HAR['counts']
act_har = RAW_HAR['activity']
raw_ids_har = RAW_HAR['session']
n_in_har = flat_har.shape[1]
n_chan_har = n_in_har + 1
n_det_har = n_chan_har * N_LEVELS
exercises_har = np.unique(act_har)

f_har, o_har, l_har = preprocess(flat_har, offsets_har, DEC_HAR, n_in_har, FS_HAR)

sess_har = infer_sessions(act_har, raw_ids_har)
n_sessions_har = int(sess_har.max()) + 1
pred_sess = run_leave_group_out(f_har, o_har, l_har, y_har, sess_har, n_chan_har, n_det_har,
                                n_splits=n_sessions_har)
m_sess = score(pred_sess, y_har)

print(f'{len(y_har)} sets | {len(exercises_har)} exercises | {n_sessions_har} inferred '
      f'sessions | {n_det_har} detectors')
report('HAR_Crossfit', 'leave-session-out  ', m_sess)

446 sets | 10 exercises | 50 inferred sessions | 156 detectors
HAR_Crossfit | leave-session-out   | exact 75.78% | +-1 94.62% | +-2 97.09% | MAE 0.437 | RMSE 1.385


In [5]:

FS_CARA = 50.0
MAX_SECONDS_CARA = 120.0
MIN_SECONDS_CARA = 3.0
DEC_CARA = round(FS_CARA / TARGET_FS_WORK)

ROOT_CARA = str(next(p for p in sorted(Path('/kaggle/input').rglob('CaRa dataset'))
                     if p.is_dir()))


def load_csv_cara(path):
    d = pd.read_csv(path, header=None).values.astype(np.float64)
    if d.ndim != 2 or d.shape[1] < 7:
        return None
    d = d[np.all(np.isfinite(d[:, :7]), axis=1)]
    if len(d) < 20:
        return None
    t, x = d[:, 0], d[:, 1:7]
    order = np.argsort(t, kind='stable')
    t, x = t[order], x[order]
    _, u = np.unique(t, return_index=True)
    t, x = t[np.sort(u)], x[np.sort(u)]
    if len(t) < 20 or t[-1] <= t[0]:
        return None
    return t / 1000.0, x


def resample_cara(t, x, fs=FS_CARA):
    """rates vary 100-333 Hz across file"""
    n = int(np.floor((t[-1] - t[0]) * fs)) + 1
    grid = t[0] + np.arange(n) / fs
    return np.stack([np.interp(grid, t, x[:, c]) for c in range(x.shape[1])], axis=1)


def build_cara(root, split):
    files = sorted(glob.glob(os.path.join(root, split, '**', '*.csv'), recursive=True))
    parts, lengths, counts, acts = [], [], [], []
    for f in files:
        got = load_csv_cara(f)
        if got is None:
            continue
        t, x = got
        if not (MIN_SECONDS_CARA <= t[-1] - t[0] <= MAX_SECONDS_CARA):
            continue
        sig = resample_cara(t, x)
        if len(sig) < 32:
            continue
        base = os.path.basename(f)[:-4].split('_')
        parts.append(sig.astype(np.float32))
        lengths.append(len(sig))
        counts.append(int(base[-1]))
        acts.append(os.path.relpath(f, os.path.join(root, split)).split(os.sep)[0])
    lengths = np.asarray(lengths, np.int64)
    offsets = np.zeros(len(lengths) + 1, np.int64)
    offsets[1:] = np.cumsum(lengths)
    return (np.concatenate(parts, 0), offsets, np.asarray(counts, np.int64), np.asarray(acts))


flat_tr, off_tr, y_tr, act_tr = build_cara(ROOT_CARA, 'train')
flat_va, off_va, y_va, act_va = build_cara(ROOT_CARA, 'val')

n_in_cara = flat_tr.shape[1]
n_chan_cara = n_in_cara + 1
n_det_cara = n_chan_cara * N_LEVELS

flat_cara = np.concatenate([flat_tr, flat_va], 0)
lengths_cara = np.concatenate([off_tr[1:] - off_tr[:-1], off_va[1:] - off_va[:-1]])
offsets_cara = np.zeros(len(lengths_cara) + 1, np.int64)
offsets_cara[1:] = np.cumsum(lengths_cara)
y_cara = np.concatenate([y_tr, y_va])
act_cara = np.concatenate([act_tr, act_va])
TR_CARA = np.arange(len(y_tr))
VA_CARA = np.arange(len(y_tr), len(y_cara))

f_cara, o_cara, l_cara = preprocess(flat_cara, offsets_cara, DEC_CARA, n_in_cara, FS_CARA)

ifit_cara, isel_cara = inner_split(act_cara, TR_CARA)
cfg_cara = select_config(f_cara, o_cara, l_cara, y_cara, act_cara, n_chan_cara, n_det_cara,
                         ifit_cara, isel_cara)
pred_cara = fit_predict(f_cara, o_cara, l_cara, y_cara, n_chan_cara, n_det_cara,
                        TR_CARA, VA_CARA, cfg_cara)
m_cara = score(pred_cara, y_cara[VA_CARA])

print(f'train {len(y_tr)} recordings / {len(set(act_tr))} classes | '
      f'val {len(y_va)} recordings / {len(set(act_va))} classes | {n_det_cara} detectors')
report('CaRaCount', 'held-out action classes', m_cara)


train 1140 recordings / 35 classes | val 513 recordings / 15 classes | 84 detectors
CaRaCount | held-out action classes | exact 24.56% | +-1 48.93% | +-2 62.18% | MAE 4.542 | RMSE 9.902


In [6]:

rows = [
    ('Microsoft/RecoFit', 'leave-activity-out', m_ms),
    ('HAR_Crossfit', 'leave-session-out', m_sess),
    ('CaRaCount', 'held-out action classes', m_cara),
]

print(f'{"dataset":<20} {"protocol":<26} {"exact":>7} {"+-1":>7} {"+-2":>7} {"MAE":>7} {"RMSE":>7}')
for name, proto, m in rows:
    print(f'{name:<20} {proto:<26} {m["exact"]:>6.2%} {m["w1"]:>6.2%} {m["w2"]:>6.2%} '
          f'{m["mae"]:>7.3f} {m["rmse"]:>7.3f}')

print(f'\nmean over {len(rows)} evaluations: '
      f'exact {np.mean([m["exact"] for _, _, m in rows]):.2%} | '
      f'+-1 {np.mean([m["w1"] for _, _, m in rows]):.2%} | '
      f'+-2 {np.mean([m["w2"] for _, _, m in rows]):.2%} | '
      f'MAE {np.mean([m["mae"] for _, _, m in rows]):.3f}')


dataset              protocol                     exact     +-1     +-2     MAE    RMSE
Microsoft/RecoFit    leave-activity-out         59.66% 82.38% 85.86%   1.985   5.212
HAR_Crossfit         leave-session-out          75.78% 94.62% 97.09%   0.437   1.385
CaRaCount            held-out action classes    24.56% 48.93% 62.18%   4.542   9.902

mean over 3 evaluations: exact 53.34% | +-1 75.31% | +-2 81.71% | MAE 2.321
